# 4. Five designs, and objects that come from a database

Sections 7 and 8 of `demo/dci_demo.py`, and section 8 again with instance binding. The designs are compared in [APPROACHES.md](https://github.com/CesarBallardini/dci-in-python/blob/main/APPROACHES.md#five-ways-to-write-the-transfer), and the persistence finding is under [Whether a persisted object can play a role](https://github.com/CesarBallardini/dci-in-python/blob/main/APPROACHES.md#whether-a-persisted-object-can-play-a-role).

Section 8 uses SQLAlchemy, pydantic and attrs. In the browser they come from the Pyodide distribution, whose versions can differ from `uv.lock`, so the first cell prints them beside the runtime.

The first cell installs `banking` when the notebook runs in the browser. On the desktop it is already installed in the project environment, and the cell only reports the runtime.

In [ ]:
import sys

if sys.platform == 'emscripten':  # in the browser: this repository's wheel, served with the site
    import piplite

    await piplite.install(['banking', 'sqlalchemy', 'pydantic', 'attrs'])

import attrs
import pydantic
import sqlalchemy

print('Python', sys.version.split()[0], 'on', sys.platform)
print('sqlalchemy', sqlalchemy.__version__, '| pydantic', pydantic.VERSION, '| attrs', attrs.__version__)

In [ ]:
from dataclasses import dataclass

from pydantic import BaseModel
from sqlalchemy import Column, String, Table, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, clear_mappers, mapped_column, registry
from sqlalchemy.types import TypeDecorator

from banking.alternatives import SourceWrapper, WrappedTransfer
from banking.contexts.money_transfer import InstanceMoneyTransferContext, MoneyTransferContext
from banking.data.account import Account
from banking.data.money import Money


def d(title):
    print(f'\n{"=" * 62}\n{title}\n{"=" * 62}')

## The role as a wrapper

The role implemented as an object that holds the account and forwards to it. It moves the money and fails none of the cases in notebook 3, and the object playing the role is not the account.

In [ ]:
d('7. The alternative without role binding: the role as a wrapper')
c = Account('Charlie', Money(500))
dd = Account('Dave', Money(0))
WrappedTransfer(c, dd, Money(50)).execute()
print(c, dd)
w = SourceWrapper(c, WrappedTransfer(c, dd, Money(0)))
print('w is c ->', w is c, '| isinstance(w, Account) ->', isinstance(w, Account))

## Five ways to write the domain object

In an existing system the account was not built by the caller: it came from a database or a model. The five classes below share no ancestor, and the same three roles are bound to each of them unchanged. The two SQLAlchemy variants share `MoneyType`, the piece of infrastructure that keeps the domain class from learning that a database exists.

In [ ]:
class MoneyType(TypeDecorator):
    impl = String
    cache_ok = True

    def process_bind_param(self, value, dialect):
        return None if value is None else f'{value.amount}:{value.currency}'

    def process_result_value(self, value, dialect):
        if value is None:
            return None
        amount, currency = value.split(':')
        return Money(int(amount), currency)


# (a) imperative: the real Account, unmodified, with the table described separately.
reg = registry()
reg.map_imperatively(
    Account,
    Table(
        'accounts',
        reg.metadata,
        Column('holder', String, primary_key=True),
        Column('balance', MoneyType, nullable=False),
    ),
)
engine = create_engine('sqlite://')
reg.metadata.create_all(engine)
with Session(engine) as session:
    session.add(Account('Alice', Money(1000)))
    session.add(Account('Bob', Money(0)))
    session.commit()


def imperative_pair():
    session = Session(engine)
    return session.get(Account, 'Alice'), session.get(Account, 'Bob')


# (b) declarative: the framework becomes part of the domain class.
class Base(DeclarativeBase):
    pass


class DeclarativeAccount(Base):
    __tablename__ = 'declarative_accounts'
    holder: Mapped[str] = mapped_column(primary_key=True)
    balance: Mapped[Money] = mapped_column(MoneyType)

    def debit(self, m):
        self.balance = self.balance - m

    def credit(self, m):
        self.balance = self.balance + m


class PydanticAccount(BaseModel):
    holder: str
    balance: Money
    model_config = {'arbitrary_types_allowed': True}

    def debit(self, m):
        self.balance = self.balance - m

    def credit(self, m):
        self.balance = self.balance + m


@attrs.define  # slots=True is the default
class AttrsAccount:
    holder: str
    balance: Money

    def debit(self, m):
        self.balance = self.balance - m

    def credit(self, m):
        self.balance = self.balance + m


@dataclass(frozen=True)
class FrozenAccount:
    holder: str
    balance: Money

    def debit(self, m):
        object.__setattr__(self, 'balance', self.balance - m)

    def credit(self, m):
        object.__setattr__(self, 'balance', self.balance + m)


STYLES = [
    ('SQLAlchemy imperative (from the DB)', imperative_pair),
    (
        'SQLAlchemy declarative',
        lambda: (
            DeclarativeAccount(holder='Alice', balance=Money(1000)),
            DeclarativeAccount(holder='Bob', balance=Money(0)),
        ),
    ),
    (
        'pydantic v2 BaseModel',
        lambda: (
            PydanticAccount(holder='Alice', balance=Money(1000)),
            PydanticAccount(holder='Bob', balance=Money(0)),
        ),
    ),
    ('attrs @define (slots by default)', lambda: (AttrsAccount('Alice', Money(1000)), AttrsAccount('Bob', Money(0)))),
    ('@dataclass(frozen=True)', lambda: (FrozenAccount('Alice', Money(1000)), FrozenAccount('Bob', Money(0)))),
]


def check(style, build_pair, binder=MoneyTransferContext):
    """Build two accounts in the given style and attempt a full transfer."""
    try:
        source, destination = build_pair()
        binder(source, destination, Money(250)).execute()
        print(f'  WORKS  {style:<38} -> {source.balance.amount} / {destination.balance.amount}')
    except Exception as exc:
        detail = str(exc).replace('\n', ' ')
        print(f'  BREAKS {style:<38} -> {type(exc).__name__}: {detail[:58]}')

In [ ]:
d('8. Five ways to write the domain object, five outcomes')
for style, build_pair in STYLES:
    check(style, build_pair)

print()
print('  Class binding works only on plain Python domain objects,')
print('  which is what DCI asks for in any case.')

## The same five, with instance binding

`InstanceContext` never touches `__class__`, so the layout of the class does not matter to it. It still needs an instance dictionary to write the role's methods into, which a slotted class does not have.

In [ ]:
d('8b. The same five, with instance binding')
for style, build_pair in STYLES:
    check(style, build_pair, InstanceMoneyTransferContext)
clear_mappers()